# ATLAS v3.0 → Kaggle Dataset Ingestion
**For full dataset (1453 sessions).**

## Background
The full ATLAS v3.0 dataset (~1453 sessions) is now available.
- Raw size: ~8GB (compressed), ~16GB (extracted)
- Preprocessed: ~10-12GB
- Total: ~26-30GB which exceeds Kaggle's 20GB working directory limit

## Solution
- Extract to `/kaggle/temp` (~100GB available)
- Output organized data to `/kaggle/temp/atlas3_raw`
- Symlink from `/kaggle/working/atlas3_raw` for easy access

## Changes from v2.0 notebook
- Uses /kaggle/temp for extraction (16GB free space)
- Updated to handle 1453 sessions (was 955)
- Added safety filter for tar extraction


## 0. Imports

In [1]:
import os
import hashlib
import tarfile
import zipfile
import shutil
import json
import subprocess
from pathlib import Path
import pandas as pd
from urllib.request import urlretrieve
from collections import defaultdict

print(f"Working dir: {Path.cwd()}")

Working dir: /kaggle/working


## 1. Config â€” edit this cell only

In [ ]:
# â”€â”€ EDIT BELOW â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
NITRC_URL = "https://fcp-indi.s3.us-east-1.amazonaws.com/data/Projects/INDI/ATLAS/R3.0/atlas3_training_raw.tar.gz"   # full URL including any token
ARCHIVE_FILENAME = "atlas3_training_raw.tar.gz"         # The raw encrypted file name
DECRYPTION_KEY = "bLw,A>?jJ6j6KnV" #"bLw,A>?jJ6j6KnV"                   # Put your OpenSSL password here
# â”€â”€ END EDIT â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

# Space management for full v3.0 dataset (1453 sessions, ~16GB extracted):
# - Working directory: 20GB limit (too small for all steps combined)
# - Solution: Use /kaggle/temp for extraction (~100GB available)
import os
if os.environ.get("KAGGLE_KERNEL_RUN_TYPE") == "Batch":
    # Running on Kaggle - use temp directory
    WORK_DIR     = Path("/kaggle/working")
    EXTRACT_DIR  = Path("/kaggle/temp/atlas3_extracted")
    OUTPUT_DIR   = Path("/kaggle/temp/atlas3_raw")        # Store organized output in temp
    # Symlink to working for easy access
    WORK_LINK = WORK_DIR / "atlas3_raw"
else:
    # Running locally - use working dir
    WORK_DIR     = Path("/kaggle/working")
    EXTRACT_DIR  = WORK_DIR / "atlas3_extracted"
    OUTPUT_DIR   = WORK_DIR / "atlas3_raw"

ARCHIVE_PATH = WORK_DIR / ARCHIVE_FILENAME
DECRYPTED_FILENAME = "ATLAS_R3_raw.tar.gz"             # The output clean filename after decryption
DECRYPTED_PATH = WORK_DIR / DECRYPTED_FILENAME

# Create directories
for d in [EXTRACT_DIR, OUTPUT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Create symlink from working dir to temp (for easier access)
if WORK_LINK.exists():
    WORK_LINK.unlink()
WORK_LINK.symlink_to(OUTPUT_DIR, target_is_directory=True)

print("Config OK")
print(f"  Archive   : {ARCHIVE_PATH}")
print(f"  Extract to: {EXTRACT_DIR}")
print(f"  Output    : {OUTPUT_DIR}")

## 2. Download

In [4]:
def _progress(block_num, block_size, total_size):
    """Minimal download progress indicator."""
    downloaded = block_num * block_size
    if total_size > 0:
        pct = min(downloaded / total_size * 100, 100)
        if block_num % 10000 == 0:
            print(f"  {pct:.1f}%  ({downloaded / 1e6:.0f} MB / {total_size / 1e6:.0f} MB)",
                  flush=True)

if ARCHIVE_PATH.exists():
    print(f"Archive already present ({ARCHIVE_PATH.stat().st_size / 1e6:.0f} MB), skipping download.")
else:
    print(f"Downloading {ARCHIVE_FILENAME} ...")
    urlretrieve(NITRC_URL, ARCHIVE_PATH, reporthook=_progress)
    print(f"Download complete: {ARCHIVE_PATH.stat().st_size / 1e6:.0f} MB")

  0.0%  (0 MB / 8244 MB)
  1.0%  (82 MB / 8244 MB)
  2.0%  (164 MB / 8244 MB)
  3.0%  (246 MB / 8244 MB)
  4.0%  (328 MB / 8244 MB)
  5.0%  (410 MB / 8244 MB)
  6.0%  (492 MB / 8244 MB)
  7.0%  (573 MB / 8244 MB)
  7.9%  (655 MB / 8244 MB)
  8.9%  (737 MB / 8244 MB)
  9.9%  (819 MB / 8244 MB)
  10.9%  (901 MB / 8244 MB)
  11.9%  (983 MB / 8244 MB)
  12.9%  (1065 MB / 8244 MB)
  13.9%  (1147 MB / 8244 MB)
  14.9%  (1229 MB / 8244 MB)
  15.9%  (1311 MB / 8244 MB)
  16.9%  (1393 MB / 8244 MB)
  17.9%  (1475 MB / 8244 MB)
  18.9%  (1556 MB / 8244 MB)
  19.9%  (1638 MB / 8244 MB)
  20.9%  (1720 MB / 8244 MB)
  21.9%  (1802 MB / 8244 MB)
  22.9%  (1884 MB / 8244 MB)
  23.8%  (1966 MB / 8244 MB)
  24.8%  (2048 MB / 8244 MB)
  25.8%  (2130 MB / 8244 MB)
  26.8%  (2212 MB / 8244 MB)
  27.8%  (2294 MB / 8244 MB)
  28.8%  (2376 MB / 8244 MB)
  29.8%  (2458 MB / 8244 MB)
  30.8%  (2540 MB / 8244 MB)
  31.8%  (2621 MB / 8244 MB)
  32.8%  (2703 MB / 8244 MB)
  33.8%  (2785 MB / 8244 MB)
  34.8%  (28

## 3. Decrypt and Extract

In [ ]:
def decrypt_file(encrypted_path: Path, decrypted_path: Path, key: str) -> None:
    """Decrypts OpenSSL AES-256-CBC base64 file using the given key."""
    if decrypted_path.exists():
        print("Decrypted file already exists. Skipping decryption step.")
        return

    print(f"Decrypting {encrypted_path.name} via OpenSSL...")
    # OpenSSL command provided by ISLES 2026. -k specifies the password programmatically.
    cmd = [
        "openssl", "aes-256-cbc", "-md", "sha256", "-d", "-a",
        "-in", str(encrypted_path),
        "-out", str(decrypted_path),
        "-k", key
    ]
    
    # Run the system shell execution cleanly
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        if decrypted_path.exists():
            os.remove(decrypted_path) # Clean up partial failures
        raise RuntimeError(f"OpenSSL Decryption failed! Check your password. Error:\n{result.stderr}")
    print("Decryption complete.")

def extract_archive(archive_path: Path, dest: Path) -> None:
    """Extract .zip or .tar.gz archive to dest. Uses filter='data' for safety."""
    filename = archive_path.name.lower()
    print(f"Extracting {archive_path.name} â†’ {dest} ...")
    
    if filename.endswith(".zip"):
        with zipfile.ZipFile(archive_path, "r") as zf:
            zf.extractall(dest)
    elif filename.endswith((".tar.gz", ".tgz")):
        with tarfile.open(archive_path, "r:gz", filter="data") as tf:
            tf.extractall(dest)
    elif filename.endswith(".tar"):
        with tarfile.open(archive_path, "r:", filter="data") as tf:
            tf.extractall(dest)
    else:
        # Fallback to get any actual extension left over for logging
        suffix = "".join(archive_path.suffixes)
        raise ValueError(f"Unsupported archive format: {suffix}")
        
    print("Extraction complete.")

## 4. Locate the Raw Data folder
ISLES'26 requires the **Raw Data** folder specifically â€” not the standardised versions.

In [6]:
def find_raw_data_root(search_root: Path) -> Path:
    """
    Walk the extracted tree and return the path of the folder
    whose name contains 'Raw' (case-insensitive).
    Falls back to the root itself if no such folder is found.
    """
    for p in sorted(search_root.rglob("*")):
        if p.is_dir() and "raw" in p.name.lower():
            print(f"Found Raw Data folder: {p}")
            return p
    print(f"WARNING: No 'Raw' folder found â€” using extract root: {search_root}")
    return search_root


RAW_DATA_ROOT = find_raw_data_root(EXTRACT_DIR)

# Quick top-level peek
top_level = sorted(RAW_DATA_ROOT.iterdir())[:10]
print(f"\nTop-level contents ({len(top_level)} shown):")
for p in top_level:
    print(f"  {'DIR' if p.is_dir() else 'FILE'}  {p.name}")

Found Raw Data folder: /kaggle/working/atlas3_extracted/ATLAS3_Training_Raw

Top-level contents (10 shown):
  FILE  .DS_Store
  DIR  R001
  DIR  R002
  DIR  R003
  DIR  R004
  DIR  R005
  DIR  R008
  DIR  R009
  DIR  R010
  DIR  R011


## 4b. Inspect raw structure (run once to understand depth)

In [7]:
# Print a 3-level tree to understand the actual folder hierarchy
def print_tree(root: Path, max_depth: int = 3, max_items: int = 4) -> None:
    def _recurse(path, depth):
        if depth > max_depth:
            return
        children = sorted(path.iterdir())[:max_items]
        for child in children:
            indent = "  " * depth
            tag = "DIR " if child.is_dir() else "FILE"
            print(f"{indent}{tag}  {child.name}")
            if child.is_dir():
                _recurse(child, depth + 1)
        remainder = len(sorted(path.iterdir())) - max_items
        if remainder > 0:
            print(f"{'  ' * (depth)}  ... and {remainder} more")
    _recurse(root, 0)

print_tree(RAW_DATA_ROOT, max_depth=4, max_items=3)

FILE  .DS_Store
DIR   R001
  FILE  .DS_Store
  DIR   sub-r001s001
    DIR   ses-1
      DIR   anat
        FILE  sub-r001s001_ses-1_metadata.csv
        FILE  sub-r001s001_ses-1_space-orig_desc-brain_T1w.nii.gz
        FILE  sub-r001s001_ses-1_space-orig_label-lesion_desc-T1lesion_mask.nii.gz
  DIR   sub-r001s002
    DIR   ses-1
      DIR   anat
        FILE  sub-r001s002_ses-1_metadata.csv
        FILE  sub-r001s002_ses-1_space-orig_desc-brain_T1w.nii.gz
        FILE  sub-r001s002_ses-1_space-orig_label-lesion_desc-T1lesion_mask.nii.gz
    ... and 37 more
DIR   R002
  DIR   sub-r002s001
    DIR   ses-1
      DIR   anat
        FILE  sub-r002s001_ses-1_metadata.csv
        FILE  sub-r002s001_ses-1_space-orig_desc-brain_T1w.nii.gz
        FILE  sub-r002s001_ses-1_space-orig_label-lesion_desc-T1lesion_mask.nii.gz
  DIR   sub-r002s002
    DIR   ses-1
      DIR   anat
        FILE  sub-r002s002_ses-1_metadata.csv
        FILE  sub-r002s002_ses-1_space-orig_desc-brain_T1w.nii.gz
        FIL

## 4c. Locate root-level metadata file

In [8]:
# ATLAS R2 metadata is typically a CSV/TSV at the archive root (above Training_Raw)
# Search one level above RAW_DATA_ROOT and also within it

def find_metadata_file(search_root: Path) -> Path | None:
    """Find the first CSV or TSV that looks like a participant/metadata table."""
    candidates = []
    for ext in ("*.csv", "*.tsv", "*.xlsx"):
        candidates.extend(search_root.rglob(ext))
    if not candidates:
        return None
    # Prefer files with 'participant' or 'metadata' or 'demographic' in name
    priority_keys = ["participant", "metadata", "demographic", "subject", "atlas"]
    for key in priority_keys:
        for c in candidates:
            if key in c.name.lower():
                return c
    return candidates[0]  # fallback: first found

# Search from the extract root (one level above Training_Raw)
# META_FILE = find_metadata_file(EXTRACT_DIR)

# if META_FILE:
#     print(f"Metadata file found: {META_FILE}")
#     df_meta = pd.read_csv(META_FILE, sep=None, engine="python")  # auto-detect sep
#     print(f"Shape: {df_meta.shape}")
#     print(df_meta.head())
#     print(f"Columns: {list(df_meta.columns)}")
# else:
#     print("WARNING: No metadata CSV/TSV found. Will proceed without metadata.")
#     df_meta = None

In [ ]:
meta_records = []
for csv_file in sorted(RAW_DATA_ROOT.rglob("*metadata.csv")):
    df = pd.read_csv(csv_file)
    meta_records.append(df)

df_meta = pd.concat([df for df in meta_records if not df.empty],
                    ignore_index=True)
# assert len(df_meta) == 955, f"Expected 955 metadata rows, got {len(df_meta)}"

df_meta["CHRONICITY_DERIVED"] = df_meta["DAYS_POST_STROKE"].apply(
    lambda d: "unknown" if pd.isna(d) else
              "acute"   if d <= 7   else
              "subacute" if d <= 90 else "chronic"
)

metadata_dir = OUTPUT_DIR / "metadata"
metadata_dir.mkdir(parents=True, exist_ok=True)

df_meta.to_csv(metadata_dir / "metadata.csv", index=False)
print(f"âœ“ Metadata saved: {df_meta.shape}")
print(df_meta["CHRONICITY_DERIVED"].value_counts())
print(f"DAYS_POST_STROKE nulls: {df_meta['DAYS_POST_STROKE'].isna().sum()}")

âœ“ Metadata saved: (1451, 6)
CHRONICITY_DERIVED
chronic     762
unknown     339
subacute    258
acute        92
Name: count, dtype: int64
DAYS_POST_STROKE nulls: 339


/tmp/ipykernel_57/1107519458.py:6: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_meta = pd.concat(meta_records, ignore_index=True)


## 5. Validate and inventory the dataset

In [11]:
def inventory_dataset(raw_root: Path) -> dict:
    """
    Walk ATLAS R2 hierarchy: raw_root/<site>/<subject>/<session>/<files>
    
    For each leaf session directory, collect:
      - All T1w NIfTI files
      - All lesion/mask NIfTI files (may be multiple raters)
      - Any JSON sidecar

    Returns a list of session-level records keyed by unique ID: site_subject_session.
    """
    records = []
    stats = defaultdict(int)

    # Determine hierarchy depth by inspecting a sample path
    # Heuristic: a session dir is the deepest dir that contains .nii files directly
    def is_session_dir(path: Path) -> bool:
        return any(f.suffix in (".nii", ".gz") for f in path.iterdir() if f.is_file())

    for nii_file in sorted(raw_root.rglob("*.nii.gz")):
        session_dir = nii_file.parent
        # Build relative path parts to extract site/subject/session labels
        rel_parts = session_dir.relative_to(raw_root).parts  # e.g. ('R001', 'sub-r001s001', 'ses-1')

        site    = rel_parts[0] if len(rel_parts) >= 1 else "unknown"
        subject = rel_parts[1] if len(rel_parts) >= 2 else "unknown"
        session = rel_parts[2] if len(rel_parts) >= 3 else "ses-01"
        uid     = f"{site}__{subject}__{session}"

        # Skip if already recorded this session
        if any(r["uid"] == uid for r in records):
            continue

        # Collect all files for this session dir
        all_niis = sorted(session_dir.glob("*.nii.gz"))
        t1_files   = [f for f in all_niis if "T1w" in f.name or "t1w" in f.name.lower()]
        mask_files = [f for f in all_niis
                      if any(k in f.name.lower() for k in ("lesion", "mask", "label"))]
        json_files = sorted(session_dir.glob("*.json"))

        record = {
            "uid":        uid,
            "site":       site,
            "subject":    subject,
            "session":    session,
            "t1_files":   [str(f) for f in t1_files],
            "mask_files": [str(f) for f in mask_files],
            "json_files": [str(f) for f in json_files],
            "has_t1":     len(t1_files) == 1,       # exactly 1 expected
            "has_mask":   len(mask_files) >= 1,
            "n_raters":   len(mask_files),
            "t1_ambiguous": len(t1_files) != 1,
        }
        records.append(record)

        stats["total"] += 1
        if record["has_t1"]:   stats["ok_t1"] += 1
        if record["has_mask"]: stats["ok_mask"] += 1
        if record["t1_ambiguous"]: stats["ambiguous_t1"] += 1

    # Per-site summary
    site_counts = defaultdict(int)
    for r in records:
        site_counts[r["site"]] += 1

    return {
        "records": records,
        "stats": dict(stats),
        "site_counts": dict(sorted(site_counts.items())),
    }


inventory = inventory_dataset(RAW_DATA_ROOT)
stats = inventory["stats"]

print("=" * 50)
print(f"Total sessions    : {stats.get('total', 0)}")
print(f"With T1w (exact 1): {stats.get('ok_t1', 0)}")
print(f"With mask         : {stats.get('ok_mask', 0)}")
print(f"Ambiguous T1w     : {stats.get('ambiguous_t1', 0)}")
print("\nSessions per site:")
for site, count in inventory["site_counts"].items():
    print(f"  {site:10s}: {count}")

# Flag any sessions with unexpected rater counts
rater_counts = defaultdict(list)
for r in inventory["records"]:
    rater_counts[r["n_raters"]].append(r["uid"])
print("\nRater mask distribution:")
for n, uids in sorted(rater_counts.items()):
    print(f"  {n} mask(s): {len(uids)} sessions")
    if n not in (1, 2):  # flag unexpected counts
        print(f"    WARNING â€” unexpected rater count. Example: {uids[:3]}")

Total sessions    : 1453
With T1w (exact 1): 1453
With mask         : 1453
Ambiguous T1w     : 0

Sessions per site:
  R001      : 39
  R002      : 12
  R003      : 15
  R004      : 37
  R005      : 37
  R008      : 7
  R009      : 111
  R010      : 26
  R011      : 29
  R014      : 9
  R015      : 23
  R016      : 1
  R017      : 16
  R018      : 11
  R019      : 14
  R020      : 1
  R023      : 15
  R024      : 21
  R027      : 32
  R028      : 26
  R029      : 10
  R030      : 15
  R031      : 37
  R032      : 37
  R033      : 2
  R034      : 24
  R035      : 15
  R038      : 94
  R039      : 4
  R040      : 86
  R041      : 3
  R042      : 35
  R044      : 4
  R045      : 4
  R046      : 13
  R047      : 48
  R048      : 44
  R049      : 24
  R050      : 15
  R052      : 32
  R053      : 5
  R056      : 15
  R057      : 14
  R058      : 35
  R059      : 38
  R061      : 15
  R062      : 8
  R063      : 1
  R064      : 6
  R065      : 18
  R067      : 27
  R069      : 40
  R070     

## 6. Organise into clean output structure
Copies (does not move) files into a flat, predictable layout:
```
atlas_raw/
  images/   <subject_id>_T1w.nii.gz
  masks/    <subject_id>_mask.nii.gz
  metadata/ <subject_id>.json
  inventory.json
```

In [12]:
def organise_output(inventory: dict, output_dir: Path, df_meta: pd.DataFrame | None) -> None:
    """
    Flatten the ATLAS R2 tree into:
      images/    {uid}_T1w.nii.gz
      masks/     {uid}_rater1.nii.gz  (and _rater2 if present)
      metadata/  metadata.csv         (root-level metadata, if found)
      inventory.json

    Skips sessions missing T1w or all masks.
    Keeps both rater masks â€” rater1/rater2 averaging is handled at training time.
    """
    images_dir   = output_dir / "images"
    masks_dir    = output_dir / "masks"
    metadata_dir = output_dir / "metadata"

    for d in [images_dir, masks_dir, metadata_dir]:
        d.mkdir(parents=True, exist_ok=True)

    copied = skipped = 0
    skip_reasons = defaultdict(list)

    for rec in inventory["records"]:
        uid = rec["uid"]

        # --- Guard: must have exactly 1 T1w and at least 1 mask ---
        if rec["t1_ambiguous"]:
            skip_reasons["ambiguous_t1"].append(uid)
            skipped += 1
            continue
        if not rec["has_t1"]:
            skip_reasons["missing_t1"].append(uid)
            skipped += 1
            continue
        if not rec["has_mask"]:
            skip_reasons["missing_mask"].append(uid)
            skipped += 1
            continue

        # --- Copy T1w ---
        t1_dst = images_dir / f"{uid}_T1w.nii.gz"
        if not t1_dst.exists():
            shutil.copy2(rec["t1_files"][0], t1_dst)

        # --- Copy masks (one per rater) ---
        for i, mask_src in enumerate(rec["mask_files"], start=1):
            mask_dst = masks_dir / f"{uid}_rater{i}.nii.gz"
            if not mask_dst.exists():
                shutil.copy2(mask_src, mask_dst)

        # --- Copy JSON sidecar if present ---
        if rec["json_files"]:
            json_dst = metadata_dir / f"{uid}.json"
            if not json_dst.exists():
                shutil.copy2(rec["json_files"][0], json_dst)

        copied += 1

    # --- Save root-level metadata CSV ---
    if df_meta is not None:
        df_meta.to_csv(metadata_dir / "metadata.csv", index=False)
        print(f"Metadata CSV saved ({len(df_meta)} rows).")
    else:
        print("No metadata CSV to save.")

    # --- Save inventory manifest ---
    manifest = {
        "total_sessions": len(inventory["records"]),
        "copied": copied,
        "skipped": skipped,
        "skip_reasons": {k: len(v) for k, v in skip_reasons.items()},
        "site_counts": inventory["site_counts"],
        "records": [
            {k: v for k, v in r.items() if k not in ("t1_files", "mask_files", "json_files")}
            for r in inventory["records"]
        ],
    }
    with open(output_dir / "inventory.json", "w") as f:
        json.dump(manifest, f, indent=2)

    print(f"\nOrganised: {copied} sessions copied, {skipped} skipped.")
    if skip_reasons:
        for reason, uids in skip_reasons.items():
            print(f"  Skipped ({reason}): {uids}")
    print(f"Manifest saved: {output_dir / 'inventory.json'}")


organise_output(inventory, OUTPUT_DIR, df_meta)

OSError: [Errno 28] No space left on device: '/kaggle/working/atlas3_extracted/ATLAS3_Training_Raw/R004/sub-r004s002/ses-1/anat/sub-r004s002_ses-1_space-orig_desc-brain_T1w.nii.gz' -> '/kaggle/working/atlas3_raw/images/R004__sub-r004s002__ses-1_T1w.nii.gz'

In [13]:
print(f"CHRONICITY nulls : {df_meta['CHRONICITY'].isna().sum()} / {len(df_meta)}")
print(f"DAYS nulls       : {df_meta['DAYS_POST_STROKE'].isna().sum()} / {len(df_meta)}")
print(f"ATLAS2_DATASET   : {df_meta['ATLAS2_DATASET'].value_counts().to_dict()}")

CHRONICITY nulls : 1155 / 1451
DAYS nulls       : 339 / 1451
ATLAS2_DATASET   : {'Training': 653, 'ATLAS3': 496, 'Testing': 299, 'Training_ATLAS2': 2, 'Testing_ATLAS2': 1}


In [ ]:
def derive_chronicity(days: float) -> str:
    if pd.isna(days):   return "unknown"
    if days <= 7:       return "acute"
    if days <= 90:      return "subacute"
    return "chronic"

df_meta["CHRONICITY_DERIVED"] = df_meta["DAYS_POST_STROKE"].apply(derive_chronicity)
# Fix: Single quotes inside the f-string + value_counts for scannable output
print(f"CHRONICITY derived distribution:\n{df_meta['CHRONICITY_DERIVED'].value_counts(dropna=False)}")
print(f"CHRONICITY derived categories: {df_meta['CHRONICITY_DERIVED'].unique()}")

## 7. Spot-check one sample

In [ ]:
images_dir = OUTPUT_DIR / "images"
masks_dir  = OUTPUT_DIR / "masks"

sample_images = sorted(images_dir.glob("*.nii.gz"))[:3]
assert len(sample_images) > 0, "No images found in output dir â€” check earlier steps."

print("Sample output files:")
for img in sample_images:
    mask = masks_dir / img.name.replace("_T1w", "_rater1")
    img_mb  = img.stat().st_size / 1e6
    mask_mb = mask.stat().st_size / 1e6 if mask.exists() else -1
    print(f"  {img.name:50s}  {img_mb:.1f} MB   mask: {'OK' if mask.exists() else 'MISSING'}  {mask_mb:.1f} MB")

In [ ]:
masks = sorted((OUTPUT_DIR / "masks").glob("*.nii.gz"))
images = sorted((OUTPUT_DIR / "images").glob("*.nii.gz"))

print(f"Mask count: {len(masks)}")
print(f"Image count: {len(images)}")
print("Sample:", [m.name for m in masks[:3]])

masks = sorted((OUTPUT_DIR / "masks").glob("*.nii.gz"))
images = sorted((OUTPUT_DIR / "images").glob("*.nii.gz"))

print(f"Mask count: {len(masks)}")
print(f"Image count: {len(images)}")
print("Sample:", [m.name for m in masks[:3]])

assert len(masks) == len(images) == 1453, f"Count mismatch: {len(images)} images vs {len(masks)} masks"
print(f"✓ {len(masks)} mask/image files confirmed.")

## 8. Disk usage summary

In [ ]:
import pandas as pd
from pathlib import Path

OUTPUT_DIR = Path("/kaggle/temp/atlas3_raw")

def dir_size_mb(path: Path) -> float:
    return sum(f.stat().st_size for f in path.rglob("*") if f.is_file()) / 1e6

# Re-measure all dirs fresh
images_dir   = OUTPUT_DIR / "images"
masks_dir    = OUTPUT_DIR / "masks"
metadata_dir = OUTPUT_DIR / "metadata"

n_images = len(list(images_dir.glob("*.nii.gz")))
n_masks = len(list(masks_dir.glob("*.nii.gz")))
meta_csv = metadata_dir / "metadata.csv"

print("=== Final pre-commit audit ===")
print(f"images/    : {n_images} files   {dir_size_mb(images_dir):.0f} MB")
print(f"masks/     : {n_masks} files   {dir_size_mb(masks_dir):.0f} MB")
print(f"metadata/  : {dir_size_mb(metadata_dir):.1f} MB")

# Assertions — all must pass
assert n_images == 1453,       f"Expected 1453 images, got {n_images}"
assert n_masks  == 1453,       f"Expected 1453 masks, got {n_masks}"
assert meta_csv.exists(),     "metadata.csv missing"

df = pd.read_csv(meta_csv)
assert len(df) == 1453,        f"Expected 1453 metadata rows, got {len(df)}"
assert "CHRONICITY_DERIVED" in df.columns, "CHRONICITY_DERIVED column missing"
assert "SESSION_ID"          in df.columns, "SESSION_ID column missing"
assert "DAYS_POST_STROKE"    in df.columns, "DAYS_POST_STROKE column missing"
assert "ATLAS2_DATASET"      in df.columns, "ATLAS2_DATASET column missing"

# Training-only subset check
train_df = df[df["ATLAS2_DATASET"] == "Training"]
assert len(train_df) == 1453,  f"Expected 1453 training rows, got {len(train_df)}"

print(f"  Metadata: {df.shape[0]} rows x {df.shape[1]} cols")
print(f"  Training sessions : {len(train_df)}")
print(f"  CHRONICITY_DERIVED: {df['CHRONICITY_DERIVED'].value_counts().to_dict()}")
print(f"  DAYS nulls        : {df['DAYS_POST_STROKE'].isna().sum()}")

print(f"All assertions passed.")
print(f"Next step: Process directly from {OUTPUT_DIR} using preprocessing.py")
